# Black–Scholes benchmarks vs Monte Carlo

**Start here:** This deep dive expands on `07_advanced_quant/monte_carlo_simulation.ipynb`; read the overview first for the common `TimeGrid`, `McEngine`, and pricer workflow.

Closed-form **Black–Scholes** prices anchor **European** Monte Carlo. Use them for **put–call parity** checks, **convergence** in **`num_paths`**, and **finite-difference Greeks** as a teaching reference.


## Concept

For GBM with constant volatility, **`black_scholes_call`** / **`black_scholes_put`** implement the standard formulas and **`valuations.bs_greeks`** supplies the closed-form **delta, gamma, vega, theta, rho** and **rho_q**. **Put–call parity** links calls and puts with the same strike and expiry. **Monte Carlo error** scales like \( 1/\sqrt{N} \) for crude averaging. **Bump-and-revalue** on spot and vol then serves as an independent check on those analytic Greeks — and as the only practical route for MC.

## API walkthrough

Import **`black_scholes_call`**, **`black_scholes_put`**, and **`EuropeanPricer`**. Compare **`MoneyEstimate.mean.amount`** to analytical prices.

In [1]:
import math
from finstack_quant.monte_carlo import black_scholes_call, black_scholes_put, EuropeanPricer

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.02, 0.25, 1.0
c = black_scholes_call(S, K, r, q, vol, T)
p = black_scholes_put(S, K, r, q, vol, T)
print(f"BS call: {c:.8f}")
print(f"BS put:  {p:.8f}")

forward = S * math.exp((r - q) * T)
parity_lhs = c - p
parity_rhs = math.exp(-r * T) * (forward - K)
print(f"Put-call parity LHS (C-P): {parity_lhs:.8f}")
print(f"Put-call parity RHS:       {parity_rhs:.8f}")
print(f"Parity residual:           {abs(parity_lhs - parity_rhs):.2e}")

BS call: 11.12376193
BS put:  8.22683705
Put-call parity LHS (C-P): 2.89692488
Put-call parity RHS:       2.89692488
Parity residual:           2.53e-14


## Examples

**Convergence study**: increase **`num_paths`** and tabulate **absolute error** versus **`black_scholes_call`**.

**Antithetic variates**: pair $Z$ with $-Z$ in a one-step terminal-spot estimator (see below).

**Greeks**: take the closed-form Greeks from **`valuations.bs_greeks`** and check them against central differences on **`black_scholes_call`** — minding the reporting units, which differ between the two.

In [2]:
from finstack_quant.monte_carlo import black_scholes_call, EuropeanPricer

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
analytical = black_scholes_call(S, K, r, q, vol, T)
print(f"Analytical ATM call: {analytical:.8f}")
counts = [1000, 5000, 10_000, 50_000, 100_000]
print("num_paths | MC_mean | stderr | |err|")
for n in counts:
    est = EuropeanPricer(num_paths=n, seed=123).price_call(S, K, r, q, vol, T, num_steps=252)
    m = est.mean.amount
    print(f"{n:9d} | {m:.8f} | {est.stderr:.8f} | {abs(m - analytical):.8f}")

Analytical ATM call: 10.45058357
num_paths | MC_mean | stderr | |err|
     1000 | 10.81611404 | 0.47076579 | 0.36553046
     5000 | 10.39422305 | 0.20740933 | 0.05636053
    10000 | 10.46263677 | 0.14602496 | 0.01205320


    50000 | 10.50791343 | 0.06601823 | 0.05732986


   100000 | 10.42770870 | 0.04657890 | 0.02287487


### Antithetic variates (variance reduction)

`McEngine(..., antithetic=True)` delegates pairing to Rust. `num_paths` remains the number of independent pair-mean estimators; `num_simulated_paths` reports the doubled raw simulation work. For a **European call** (monotone in the shock), averaging each pair typically cuts estimator variance.

In [3]:
from finstack_quant.monte_carlo import McEngine, TimeGrid, black_scholes_call

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
truth = black_scholes_call(S, K, r, q, vol, T)
n_estimators = 2_000
grid = TimeGrid(t_max=T, num_steps=1)

crude = McEngine(
    num_paths=n_estimators,
    time_grid=grid,
    seed=42,
    use_parallel=False,
    antithetic=False,
).price_european_call(S, K, r, q, vol)
anti = McEngine(
    num_paths=n_estimators,
    time_grid=grid,
    seed=42,
    use_parallel=False,
    antithetic=True,
).price_european_call(S, K, r, q, vol)

print(f"Black–Scholes (exact): {truth:.8f}")
print(
    f"Crude:      estimators={crude.num_paths}, simulated={crude.num_simulated_paths}, "
    f"mean={crude.mean.amount:.8f}, stderr={crude.stderr:.8f}"
)
print(
    f"Antithetic: estimators={anti.num_paths}, simulated={anti.num_simulated_paths}, "
    f"mean={anti.mean.amount:.8f}, stderr={anti.stderr:.8f}"
)

Black–Scholes (exact): 10.45058357
Crude:      estimators=2000, simulated=2000, mean=10.21135541, stderr=0.31720329
Antithetic: estimators=2000, simulated=4000, mean=10.41310759, stderr=0.16581742


### Closed-form Greeks: `bs_greeks` vs finite differences

The analytic Greeks come from Rust — `finstack_quant.valuations.bs_greeks(spot, strike, r, q, sigma, t, is_call, theta_days=365.0)` returns `delta`, `gamma`, `vega`, `theta`, `rho` and `rho_q` (the dividend-yield rho). Never re-derive them in Python: the finite differences below exist only to *verify* the binding, in the same spirit as the Monte Carlo `finite_diff_delta` / `finite_diff_gamma` checks in the next section.

**Read the units before you compare anything.** `bs_greeks` reports on trading-desk conventions, not on the mathematical per-unit basis:

| Greek | `bs_greeks` basis | Per-unit (calculus) basis |
| --- | --- | --- |
| `delta` | per 1.0 of spot | same — no scaling |
| `gamma` | per 1.0 of spot, second order | same — no scaling |
| `vega` | **per 1% (0.01) of vol** | multiply by 100 |
| `rho`, `rho_q` | **per 1% (0.01) of rate** | multiply by 100 |
| `theta` | **per day**, denominator = `theta_days` (365 default) | multiply by `theta_days` |

Pass `theta_days=252.0` when you want theta per *business* day. A vega quoted per 1% dropped into a report that expects per-1.0 is off by a factor of 100 — this is exactly the kind of silent unit mismatch that corrupts a risk file, so the cell below rescales explicitly rather than relying on the two sides happening to agree.

In [4]:
from finstack_quant.monte_carlo import black_scholes_call
from finstack_quant.valuations import bs_greeks

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0

# --- Analytic Greeks: produced by Rust, on desk conventions -------------------
g = bs_greeks(S, K, r, q, vol, T, True)  # is_call=True; theta_days defaults to 365
print("bs_greeks (as reported, desk conventions):")
print(f"  delta = {g['delta']:.8f}   per 1.0 of spot")
print(f"  gamma = {g['gamma']:.8f}   per 1.0 of spot (2nd order)")
print(f"  vega  = {g['vega']:.8f}   per 1% of vol")
print(f"  theta = {g['theta']:.8f}   per calendar day (theta_days=365)")
print(f"  rho   = {g['rho']:.8f}   per 1% of rate")
print(f"  rho_q = {g['rho_q']:.8f}   per 1% of dividend yield")

# Rescale onto the per-unit basis the finite differences below live on.
analytic = {
    "delta (per 1.0 spot)": g["delta"],
    "gamma (per 1.0 spot)": g["gamma"],
    "vega  (per 1.0 vol)": g["vega"] * 100.0,  # per 1% -> per 1.0
    "rho   (per 1.0 rate)": g["rho"] * 100.0,  # per 1% -> per 1.0
    "theta (per year)": g["theta"] * 365.0,  # per day -> per year
}

# --- Finite differences: a cross-check, NOT the source of the numbers ---------
h = 1e-4
fd = {
    "delta (per 1.0 spot)": (black_scholes_call(S + h, K, r, q, vol, T) - black_scholes_call(S - h, K, r, q, vol, T))
    / (2 * h),
    "gamma (per 1.0 spot)": (
        black_scholes_call(S + h, K, r, q, vol, T)
        - 2 * black_scholes_call(S, K, r, q, vol, T)
        + black_scholes_call(S - h, K, r, q, vol, T)
    )
    / (h * h),
    "vega  (per 1.0 vol)": (black_scholes_call(S, K, r, q, vol + h, T) - black_scholes_call(S, K, r, q, vol - h, T))
    / (2 * h),
    "rho   (per 1.0 rate)": (black_scholes_call(S, K, r + h, q, vol, T) - black_scholes_call(S, K, r - h, q, vol, T))
    / (2 * h),
    # theta = dV/dt = -dV/dT, hence the leading minus sign.
    "theta (per year)": -(black_scholes_call(S, K, r, q, vol, T + h) - black_scholes_call(S, K, r, q, vol, T - h))
    / (2 * h),
}

print("\nanalytic (rescaled) vs central finite differences on black_scholes_call:")
print(f"{'greek':<22} | {'analytic':>14} | {'finite diff':>14} | {'abs err':>10}")
for name, a in analytic.items():
    f = fd[name]
    print(f"{name:<22} | {a:>14.8f} | {f:>14.8f} | {abs(a - f):>10.2e}")

# Same theta, business-day denominator: the only thing that changes is the divisor.
theta_252 = bs_greeks(S, K, r, q, vol, T, True, theta_days=252.0)["theta"]
print(
    f"\ntheta per calendar day (365) = {g['theta']:.8f}"
    f"  |  theta per business day (252) = {theta_252:.8f}"
    f"  |  same annual theta: {g['theta'] * 365.0:.8f} vs {theta_252 * 252.0:.8f}"
)

# A coarse bump shows why FD is only a check, never the production Greek.
h_s = 1.0
coarse_delta = (black_scholes_call(S + h_s, K, r, q, vol, T) - black_scholes_call(S - h_s, K, r, q, vol, T)) / (
    2 * h_s
)
print(
    f"\nCoarse FD delta (spot bump={h_s}): {coarse_delta:.8f}"
    f"  vs analytic {g['delta']:.8f}  (truncation error {abs(coarse_delta - g['delta']):.2e})"
)

bs_greeks (as reported, desk conventions):
  delta = 0.63683065   per 1.0 of spot
  gamma = 0.01876202   per 1.0 of spot (2nd order)
  vega  = 0.37524035   per 1% of vol
  theta = -0.01757268   per calendar day (theta_days=365)
  rho   = 0.53232482   per 1% of rate
  rho_q = -0.63683065   per 1% of dividend yield

analytic (rescaled) vs central finite differences on black_scholes_call:
greek                  |       analytic |    finite diff |    abs err
delta (per 1.0 spot)   |     0.63683065 |     0.63683065 |   6.08e-12
gamma (per 1.0 spot)   |     0.01876202 |     0.01876188 |   1.36e-07
vega  (per 1.0 vol)    |    37.52403469 |    37.52403439 |   3.04e-07
rho   (per 1.0 rate)   |    53.23248155 |    53.23248077 |   7.71e-07
theta (per year)       |    -6.41402755 |    -6.41402755 |   4.92e-09

theta per calendar day (365) = -0.01757268  |  theta per business day (252) = -0.02545249  |  same annual theta: -6.41402755 vs -6.41402755

Coarse FD delta (spot bump=1.0): 0.63674469  vs a

### Monte Carlo finite-difference Greeks (with common random numbers)

`finite_diff_delta` / `finite_diff_gamma` bump-and-revalue under fresh randoms (noisy), while the `_crn` variants reuse **common random numbers** to slash the differencing variance. Each returns `(estimate, stderr)`.

In [5]:
from finstack_quant.monte_carlo import (
    finite_diff_delta,
    finite_diff_delta_crn,
    finite_diff_gamma,
    finite_diff_gamma_crn,
)

kw = dict(
    spot=100.0,
    strike=100.0,
    rate=0.05,
    div_yield=0.0,
    vol=0.20,
    expiry=1.0,
    num_paths=40_000,
    seed=7,
    option_type="call",
)
d, d_se = finite_diff_delta(**kw)
dc, dc_se = finite_diff_delta_crn(**kw)
g, g_se = finite_diff_gamma(**kw)
gc, gc_se = finite_diff_gamma_crn(**kw)
print(f"delta      = {d:.5f}  (stderr={d_se:.5f})")
print(f"delta_crn  = {dc:.5f}  (stderr={dc_se:.5f})  <- CRN cuts the stderr sharply")
print(f"gamma      = {g:.5f}  (stderr={g_se:.5f})")
print(f"gamma_crn  = {gc:.5f}  (stderr={gc_se:.5f})")

delta      = 0.63545  (stderr=0.05163)
delta_crn  = 0.63545  (stderr=0.00285)  <- CRN cuts the stderr sharply
gamma      = 0.01875  (stderr=0.17883)
gamma_crn  = 0.01875  (stderr=0.00054)


## Takeaways

- **Analytical BS** is the first sanity check for **`EuropeanPricer`**.
- **Put–call parity** catches discounting or input mismatches instantly.
- **Error vs paths** illustrates **\( 1/\sqrt{N} \)** behavior of standard errors.
- **Antithetic** pairing of normals is a simple **variance reduction** technique for smooth payoffs (e.g. European vanillas).
- **Closed-form Greeks come from `valuations.bs_greeks`**, not from Python bump-and-revalue. Finite differences are a *verification* tool here — as they are for the MC `finite_diff_*` estimators.
- **Check the units.** `bs_greeks` reports vega and both rhos **per 1%** and theta **per day** (`theta_days`, 365 by default). Finite differences are per unit and per year. Rescale explicitly whenever the two meet.
- **Bumping** is simple and robust for teaching; production often prefers **pathwise** or **likelihood ratio** Greeks for MC.